# geodetic-engine -- Well Trajectory Examples

This notebook exercises `geodetic_engine.welltrajectory` on the test data kept in the repository: the request bodies in `tests/welltrajectory/data/requests/` and the survey tables in `tests/welltrajectory/data/legacy_fixtures.json`.

A trajectory is computed in two stages. The survey (measured depth, inclination, azimuth) is reduced to offsets from the wellhead by **minimum curvature**, and those offsets are then **placed** in the trajectory CRS by one of four methods. Coordinate values are always `xy`: easting then northing, or longitude then latitude.

Plots are drawn with plotly, from the `plot` extra: `pip install 'geodetic-engine[plot]'`. Every 3D example is shown in the notebook and can be opened full screen in the browser; see [Plotting](#10-plotting).

## Contents

1. [Setup](#1-setup)
2. [A request body, end to end](#2-a-request-body-end-to-end)
3. [The same trajectory through the generic API](#3-the-same-trajectory-through-the-generic-api)
4. [Minimum curvature on its own](#4-minimum-curvature-on-its-own)
5. [The four placement methods](#5-the-four-placement-methods)
6. [Grid north, true north and projection factors](#6-grid-north-true-north-and-projection-factors)
7. [Units and interpolation](#7-units-and-interpolation)
8. [A geographic CRS](#8-a-geographic-crs)
9. [Moving to WGS 84](#9-moving-to-wgs-84)
10. [Plotting](#10-plotting)
11. [What is refused](#11-what-is-refused)

## 1. Setup

The test data is found by walking up from the working directory, so the notebook runs from `examples/` or from the repository root. Two small helpers read a request body into the generic API's `Survey` and `Wellhead`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from geodetic_engine.geodesy import (
    AmbiguousOperationError,
    UnsupportedCRSError,
    projection_factors,
)
from geodetic_engine.welltrajectory import (
    InvalidSurveyError,
    Method,
    MinimumCurvature,
    Survey,
    UnitError,
    Wellhead,
    compute_trajectory,
    open_in_browser,
    plot_trajectory,
)
from geodetic_engine.welltrajectory.utils import from_payload

ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "tests" / "welltrajectory" / "data").is_dir()
)
DATA = ROOT / "tests" / "welltrajectory" / "data"
REQUESTS = {
    path.stem: json.loads(path.read_text(encoding="utf-8"))["value"]
    for path in sorted((DATA / "requests").glob("*.json"))
}
FIXTURES = json.loads((DATA / "legacy_fixtures.json").read_text(encoding="utf-8"))


def survey_of(body: dict) -> Survey:
    """The survey stations of a request body."""
    rows = body["inputStations"]
    return Survey(
        [row["md"] for row in rows],
        [row["inclination"] for row in rows],
        [row.get("azimuth", 0.0) for row in rows],
        md_unit=body.get("unitMD") or body["unitZ"],
    )


def wellhead_of(body: dict) -> Wellhead:
    """The wellhead of a request body."""
    point = body["referencePoint"]
    return Wellhead(point["x"], point["y"], point.get("z", 0.0))


pd.set_option("display.precision", 4)
pd.set_option("display.width", 160)

pd.DataFrame(
    {
        "crs": [body["trajectoryCRS"] for body in REQUESTS.values()],
        "north": [body["azimuthReference"] for body in REQUESTS.values()],
        "method": [body.get("method") for body in REQUESTS.values()],
        "unitMD / unitZ": [
            f"{body.get('unitMD') or body['unitZ']} / {body['unitZ']}"
            for body in REQUESTS.values()
        ],
        "stations": [len(body["inputStations"]) for body in REQUESTS.values()],
        "MD_i": [body.get("MD_i") for body in REQUESTS.values()],
    },
    index=list(REQUESTS),
)

,crs,north,method,unitMD / unitZ,stations,MD_i
01_simple_vertical_well,EPSG:32631,GN,AzimuthalEquidistant,m / m,3,None
02_deviated_well_build,EPSG:32631,GN,AzimuthalEquidistant,m / m,7,None
03_horizontal_well,EPSG:32615,GN,AzimuthalEquidistant,m / m,7,None
04_s_curve_with_turn,EPSG:32631,TN,AzimuthalEquidistant,m / m,10,None
05_geographic_lmp_method,EPSG:4326,TN,LMP,m / m,5,None
06_feet_with_md_interpolation,EPSG:32615,GN,AzimuthalEquidistant,ft / ft,4,"{'md_i': [1640, 4920, 8200]}"
07_extended_reach_well,EPSG:32631,GN,AzimuthalEquidistant,m / m,9,None
08_md_interval_interpolation,EPSG:32631,GN,AzimuthalEquidistant,m / m,3,{'md_interval': 1}
09_mixed_units_md_meters_z_feet,EPSG:32631,GN,AzimuthalEquidistant,m / ft,5,{'md_interval': 250}


## 2. A request body, end to end

`from_payload()` maps a request body onto the generic API field by field and returns the trajectory (`stations`), any points `MD_i` asks for (`stations_i`), and the local CRS the offsets were read in, as a persistableReference.

Every trajectory carries `operations`, a record of what was done to produce it, in order.

In [2]:
body = REQUESTS["02_deviated_well_build"]
result = from_payload(body)
trajectory = result.stations

print(
    f"{len(trajectory)} stations in {trajectory.crs.name}, placed by {trajectory.method}"
)
print(*trajectory.operations, sep="\n")
trajectory.to_dataframe()

7 stations in WGS 84 / UTM zone 31N, placed by AzimuthalEquidistant
trajectory CRS EPSG:32631; wellhead at (500000, 6600000), elevation 50 m
grid convergence -0.000000000 deg and scale factor 0.999600000 at the wellhead
azimuths against grid north, turned onto true north by the grid convergence
minimum curvature over 7 survey stations
read offsets as WGS 84 / Azimuthal Equidistant Lat=59.53834913 Lon=3.00000000
converted from WGS 84 / Azimuthal Equidistant Lat=59.53834913 Lon=3.00000000 to WGS 84 / UTM zone 31N


,md,inclination,azimuth_true,azimuth_grid,east,north,tvd,x,y,z,dls,is_survey_station
0,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,500000.0000,6.6000e+06,50.0000,0.0,True
1,500.0,0.0,45.0,45.0,0.0000,0.0000,500.0000,500000.0000,6.6000e+06,-450.0000,0.0,True
2,800.0,15.0,45.0,45.0,27.6098,27.6098,796.5848,500027.5987,6.6000e+06,-746.5848,1.5,True
3,1100.0,30.0,45.0,45.0,108.5576,108.5576,1072.9578,500108.5141,6.6001e+06,-1022.9578,1.5,True
4,1400.0,45.0,45.0,45.0,237.3269,237.3269,1310.2847,500237.2320,6.6002e+06,-1260.2847,1.5,True
5,2000.0,45.0,45.0,45.0,537.3269,537.3269,1734.5488,500537.1120,6.6005e+06,-1684.5488,0.0,True
6,3000.0,45.0,45.0,45.0,1037.3269,1037.3269,2441.6555,501036.9120,6.6010e+06,-2391.6555,0.0,True


## 3. The same trajectory through the generic API

`compute_trajectory()` takes a `Survey`, a `Wellhead` and a CRS. The CRS can be anything `CoordinateReferenceSystem.from_user_input` accepts: an EPSG code, WKT, PROJJSON, an OSDU persistableReference, or a bound CRS.

In [3]:
generic = compute_trajectory(
    survey_of(body),
    wellhead_of(body),
    body["trajectoryCRS"],
    north=body["azimuthReference"],
    method=Method.AZIMUTHAL_EQUIDISTANT,
    z_unit=body["unitZ"],
)

print(
    "same positions as the request body:",
    np.allclose(generic.x, trajectory.x) and np.allclose(generic.y, trajectory.y),
)
print("local CRS:", generic.local_crs.name if generic.local_crs else None)

same positions as the request body: True
local CRS: WGS 84 / Azimuthal Equidistant Lat=59.53834913 Lon=3.00000000


## 4. Minimum curvature on its own

`MinimumCurvature` needs no CRS: measured depth in any length unit, angles in radians, and it returns offsets (east, north, TVD) in the unit of MD.

The survey below holds a vertical section to MD 1000, a 90° build to horizontal by MD 2000, a straight horizontal section to 3000, and a 90° horizontal turn from north to east by 5000. Each curved section is a single circle, so its geometry can be checked by hand:

- the build has radius $1000 / (\pi/2) = 636.62$ m and gains that much both northwards and in TVD;
- the turn has radius $2000 / (\pi/2) = 1273.24$ m and gains that much both eastwards and northwards.

In [4]:
table = np.array(FIXTURES["simple_survey_input"]["rows"])
model = MinimumCurvature(table[:, 0], np.radians(table[:, 1]), np.radians(table[:, 2]))
stations = model.stations

pd.DataFrame(
    {
        "md": stations.md,
        "inclination": np.degrees(stations.inclination),
        "azimuth": np.degrees(stations.azimuth),
        "east": stations.east,
        "north": stations.north,
        "tvd": stations.tvd,
        "dls [deg/30 m]": stations.dls(30),
    }
)

,md,inclination,azimuth,east,north,tvd,dls [deg/30 m]
0,0.000,0.0000,0.0,0.0000,0.0000,0.0000,0.00
1,1000.000,0.0000,0.0,0.0000,0.0000,1000.0000,0.00
2,1015.625,1.4062,0.0,0.0000,0.1917,1015.6234,2.70
3,2000.000,90.0000,0.0,0.0000,636.6198,1636.6198,2.70
4,3000.000,90.0000,0.0,0.0000,1636.6198,1636.6198,0.00
5,5000.000,90.0000,90.0,1273.2395,2909.8593,1636.6198,1.35
6,6000.000,90.0000,90.0,2273.2395,2909.8593,1636.6198,0.00


In [5]:
build, turn = 1000 / (np.pi / 2), 2000 / (np.pi / 2)
md = list(stations.md)
at = {depth: md.index(depth) for depth in (1000, 2000, 3000, 5000)}

print(
    f"build: radius {build:.3f} m, gained north {stations.north[at[2000]] - stations.north[at[1000]]:.3f}, "
    f"TVD {stations.tvd[at[2000]] - stations.tvd[at[1000]]:.3f}"
)
print(
    f"turn:  radius {turn:.3f} m, gained east {stations.east[at[5000]] - stations.east[at[3000]]:.3f}, "
    f"north {stations.north[at[5000]] - stations.north[at[3000]]:.3f}"
)

build: radius 636.620 m, gained north 636.620, TVD 636.620
turn:  radius 1273.240 m, gained east 1273.240, north 1273.240


### Points between stations stay on the arcs

`interpolate()` places a point on the circular arc of the interval it falls in, rather than on the straight line between the stations either side. Half way round the build the two differ by tens of metres.

Because every interpolated point is on the arcs, resampling a survey and running minimum curvature on the result lands on exactly the same positions.

In [6]:
point = model.interpolate([1500.0])
linear = np.array(
    [np.interp(1500.0, stations.md, stations.offsets[:, axis]) for axis in range(3)]
)

print(
    "MD 1500 on the arc:     ",
    point.offsets[0].round(3),
    f"inclination {np.degrees(point.inclination[0]):.4f} deg",
)
print("MD 1500 on the straight:", linear.round(3))
print(f"apart by {np.linalg.norm(point.offsets[0] - linear):.2f} m")

resampled = model.resample(10.0)
again = MinimumCurvature(resampled.md, resampled.inclination, resampled.azimuth)
print(
    f"\nresampled every 10 m: {len(resampled)} points, "
    f"re-running moves them at most {np.abs(again.stations.offsets - resampled.offsets).max():.1e} m"
)

MD 1500 on the arc:      [   0.     186.462 1450.158] inclination 45.0000 deg
MD 1500 on the straight: [   0.     313.355 1321.193]
apart by 180.93 m

resampled every 10 m: 602 points, re-running moves them at most 1.4e-12 m


### Against the fixture's expected table

The fixtures also hold an expected table for this survey. It agrees with the arcs to $10^{-8}$ m through the build and the horizontal section, up to MD 3000. At MD 5000 and 6000 it is wrong: it has the 90° turn move the hole east only, with a dogleg severity of zero. The arc gives $1273.24$ m both east and north, at $1.35°$/30 m.

In [7]:
spec = FIXTURES["expected_interpolated_trajectory"]
expected = pd.DataFrame(spec["rows"], columns=spec["columns"])
computed = model.interpolate(expected["md"].to_numpy())

comparison = pd.DataFrame(
    {
        "md": expected["md"],
        "east (fixture)": expected["dx"],
        "east (computed)": computed.east,
        "north (fixture)": expected["dy"],
        "north (computed)": computed.north,
        "dls (fixture)": expected["dls"],
        "dls (computed)": computed.dls(30),
        "apart [m]": np.linalg.norm(
            computed.offsets - expected[["dx", "dy", "dz"]].to_numpy(), axis=1
        ),
    }
)
comparison[comparison["md"].isin([1000, 1500, 2000, 3000, 5000, 6000])]

,md,east (fixture),east (computed),north (fixture),north (computed),dls (fixture),dls (computed),apart [m]
1,1000.0,0.0000,0.0000,0.0000,0.0000,0.0,0.00,0.0000e+00
33,1500.0,0.0000,0.0000,186.4616,186.4616,2.7,2.70,3.2015e-12
65,2000.0,0.0000,0.0000,636.6198,636.6198,2.7,2.70,1.4062e-12
66,3000.0,0.0000,0.0000,1636.6198,1636.6198,0.0,0.00,1.9293e-12
67,5000.0,1636.6198,1273.2395,1636.6198,2909.8593,0.0,1.35,1.3241e+03
68,6000.0,2636.6198,2273.2395,1636.6198,2909.8593,0.0,0.00,1.3241e+03


## 5. The four placement methods

The offsets are metres against true north at the wellhead. Each method is a different model of how that local frame sits on the curved earth:

| Method | Model |
| --- | --- |
| `AzimuthalEquidistant` | The offsets are coordinates of an azimuthal equidistant projection centred on the wellhead, converted to the CRS. |
| `GridNorthLocal` | The offsets turned by the grid convergence and scaled by the scale factor at the wellhead. |
| `ENU` | The offsets are a topocentric frame at the wellhead, taken through geocentric coordinates. |
| `LMP` | Each step laid on the ellipsoid with the radii of curvature at its own latitude and depth. |

On the extended reach well, about 8.9 km due west at 2.2 km TVD, they separate by the amounts their geometry predicts:

- `GridNorthLocal` barely differs, since the well runs along a parallel near the central meridian;
- `ENU` lands about $d\,T/R \approx 3$ m further out, the flat frame leaving the curved earth;
- `LMP` lands about as far out for the same reason, and about $d^2 \tan\varphi / 2N \approx 11$ m north: it holds each step's azimuth against its own meridian, so due west follows the parallel rather than the geodesic.

In [8]:
body = REQUESTS["07_extended_reach_well"]
by_method = {
    method: from_payload(body | {"method": str(method)}).stations for method in Method
}
reference = by_method[Method.AZIMUTHAL_EQUIDISTANT]

print(
    f"reach {np.hypot(reference.east, reference.north)[-1]:.1f} m, "
    f"TVD {reference.tvd[-1]:.1f} m, azimuth {reference.azimuth_true[-1]:.1f} deg\n"
)
pd.DataFrame(
    {
        "x at TD": [t.x[-1] for t in by_method.values()],
        "y at TD": [t.y[-1] for t in by_method.values()],
        "dx at TD [m]": [t.x[-1] - reference.x[-1] for t in by_method.values()],
        "dy at TD [m]": [t.y[-1] - reference.y[-1] for t in by_method.values()],
        "largest separation [m]": [
            np.hypot(t.x - reference.x, t.y - reference.y).max()
            for t in by_method.values()
        ],
    },
    index=[str(method) for method in by_method],
)

reach 8920.1 m, TVD 2173.4 m, azimuth 270.0 deg



,x at TD,y at TD,dx at TD [m],dy at TD [m],largest separation [m]
AzimuthalEquidistant,491083.4581,6.6000e+06,0.0000,0.0000e+00,0.0000
GridNorthLocal,491083.4610,6.6000e+06,0.0029,-1.2107e-08,0.0029
ENU,491080.4739,6.6000e+06,-2.9842,-6.1048e-06,2.9842
LMP,491081.0481,6.6000e+06,-2.4099,1.0581e+01,10.8516


In [9]:
colours = ["#2563eb", "#ea580c", "#059669", "#7c3aed"]
separation = go.Figure(
    [
        go.Scatter(
            x=placed.md,
            y=np.hypot(placed.x - reference.x, placed.y - reference.y),
            name=str(method),
            mode="lines",
            line={"width": 2.5, "color": colour},
            hovertemplate="MD %{x:.0f} m<br>%{y:.2f} m<extra>%{fullData.name}</extra>",
        )
        for (method, placed), colour in zip(by_method.items(), colours, strict=True)
    ]
)
separation.update_layout(
    template="plotly_white",
    width=960,
    height=420,
    font={"family": "Inter, 'Segoe UI', Helvetica, Arial, sans-serif", "size": 13},
    title={
        "text": "<b>Extended reach well: how far the methods separate</b>",
        "x": 0.02,
        "subtitle": {"text": "horizontal distance from the AzimuthalEquidistant path"},
    },
    xaxis={"title": "MD [m]", "tickfont": {"size": 13}},
    yaxis={"title": "separation [m]", "rangemode": "tozero", "tickfont": {"size": 13}},
    legend={"orientation": "h", "x": 0.0, "y": -0.2, "font": {"size": 13}},
    hovermode="x unified",
    margin={"l": 60, "r": 20, "t": 80, "b": 60},
)

## 6. Grid north, true north and projection factors

`geodetic_engine.geodesy.projection_factors()` gives the grid convergence $\gamma$ and the point scale factor $k$ at any point of a projected CRS. The convergence is measured from true north to grid north, clockwise positive, so $A_{grid} = A_{true} - \gamma$.

The horizontal well's wellhead sits west of its UTM zone's central meridian, where $\gamma \approx -1.47°$. Reading the same azimuths against grid north or against true north therefore sends the well in directions $1.47°$ apart, about 115 m sideways at its 4.5 km reach.

In [10]:
body = REQUESTS["03_horizontal_well"]
factors = projection_factors(
    body["trajectoryCRS"], (body["referencePoint"]["x"], body["referencePoint"]["y"])
)
print(
    f"wellhead at {factors.longitude[0]:.4f} E, {factors.latitude[0]:.4f} N: "
    f"grid convergence {factors.grid_convergence[0]:.4f} deg, scale factor {factors.scale_factor[0]:.6f}"
)
print("grid azimuth 90 is true azimuth", factors.to_true_azimuth(90.0).round(4))

grid = from_payload(body).stations
true = from_payload(body | {"azimuthReference": "TN"}).stations
pd.DataFrame(
    {
        "md": grid.md,
        "azimuth_true (read as GN)": grid.azimuth_true,
        "azimuth_grid (read as GN)": grid.azimuth_grid,
        "y (read as GN)": grid.y,
        "y (read as TN)": true.y,
        "apart [m]": np.hypot(grid.x - true.x, grid.y - true.y),
    }
)

wellhead at -94.7222 E, 58.6288 N: grid convergence -1.4706 deg, scale factor 0.999723
grid azimuth 90 is true azimuth [88.5294]


,md,azimuth_true (read as GN),azimuth_grid (read as GN),y (read as GN),y (read as TN),apart [m]
0,0.0,88.5294,90.0,6500000.0,6.5000e+06,0.0000
1,1000.0,88.5294,90.0,6500000.0,6.5000e+06,0.0000
2,1500.0,88.5294,90.0,6500000.0,6.5000e+06,3.2826
3,2000.0,88.5294,90.0,6500000.0,6.5000e+06,12.2508
4,2500.0,88.5294,90.0,6500000.0,6.5000e+06,24.5016
5,4000.0,88.5294,90.0,6500000.0,6.4999e+06,62.9885
6,6000.0,88.5294,90.0,6500000.0,6.4999e+06,114.3039


The factors can be evaluated at every station too. `to_json_dict()` gives them one entry per point, with the sign convention stated.

In [11]:
along = grid.projection_factors()
pd.DataFrame(along.to_json_dict()["points"]).drop(columns="coordinates").assign(
    md=grid.md
).set_index("md")

,longitude,latitude,grid_convergence,scale_factor,meridional_scale,areal_scale,angular_distortion
md,,,,,,,
0.0,-94.7222,58.6288,-1.4706,0.9997,0.9997,0.9994,1.2078e-06
1000.0,-94.7222,58.6288,-1.4706,0.9997,0.9997,0.9994,1.2078e-06
1500.0,-94.7200,58.6288,-1.4687,0.9997,0.9997,0.9994,8.5401e-07
2000.0,-94.7140,58.6289,-1.4635,0.9997,0.9997,0.9994,0.0000e+00
2500.0,-94.7058,58.6290,-1.4565,0.9997,0.9997,0.9994,8.5401e-07
4000.0,-94.6799,58.6293,-1.4345,0.9997,0.9997,0.9994,0.0000e+00
6000.0,-94.6455,58.6298,-1.4051,0.9997,0.9997,0.9994,0.0000e+00


## 7. Units and interpolation

Measured depth and the vertical results each have their own unit. Dogleg severity follows the MD unit: per 100 ft for a survey in feet, per 30 m otherwise.

- The feet example states MD and depths in feet, and asks for three points by MD (`md_i`).
- The mixed units example states MD in metres and depths in feet, and asks for a point every 250 m of MD (`md_interval`).

In [12]:
feet = from_payload(REQUESTS["06_feet_with_md_interpolation"])
print(
    f"MD in {feet.stations.md_unit}, depths in {feet.stations.z_unit}, DLS per {feet.stations.dls_length:g} {feet.stations.md_unit}"
)
feet.stations.to_dataframe()

MD in ft, depths in ft, DLS per 100 ft


,md,inclination,azimuth_true,azimuth_grid,east,north,tvd,x,y,z,dls,is_survey_station
0,0.0,0.0,358.5294,0.0,0.0000,0.0000,0.0000,400000.0000,6.5000e+06,0.0000,0.0000,True
1,3280.0,15.0,43.5294,45.0,294.0207,309.5144,3242.6602,400091.9835,6.5001e+06,-3242.6602,0.4573,True
2,6560.0,45.0,43.5294,45.0,1410.6750,1485.0119,6050.8864,400441.3256,6.5004e+06,-6050.8864,0.9146,True
3,9840.0,45.0,43.5294,45.0,3008.0474,3166.5593,8370.1966,400941.0589,6.5009e+06,-8370.1966,0.0000,True


In [13]:
feet.stations_i.to_dataframe()

,md,inclination,azimuth_true,azimuth_grid,east,north,tvd,x,y,z,dls,is_survey_station
0,1640.0,7.5,43.5294,45.0,73.8209,77.7110,1635.3205,400023.0947,6.5000e+06,-1635.3205,0.4573,False
1,4920.0,30.0,43.5294,45.0,725.0332,763.2395,4753.4994,400226.8245,6.5002e+06,-4753.4994,0.9146,False
2,8200.0,45.0,43.5294,45.0,2209.3612,2325.7856,7210.5415,400691.1923,6.5007e+06,-7210.5415,0.0000,False


In [14]:
body = REQUESTS["09_mixed_units_md_meters_z_feet"]
mixed = from_payload(body)
metric = from_payload(body | {"unitZ": "m"})

print(
    f"MD in {mixed.stations.md_unit}, depths in {mixed.stations.z_unit}, DLS per {mixed.stations.dls_length:g} {mixed.stations.md_unit}"
)
print(
    "TVD in feet, times 0.3048, is the TVD in metres:",
    np.allclose(mixed.stations.tvd * 0.3048, metric.stations.tvd),
)
mixed.stations_i.to_dataframe()

MD in m, depths in ft, DLS per 30 m
TVD in feet, times 0.3048, is the TVD in metres: True


,md,inclination,azimuth_true,azimuth_grid,east,north,tvd,x,y,z,dls,is_survey_station
0,0.0,0.0000,340.0000,340.0000,0.0000,0.0000,0.0000,500000.0000,6.5000e+06,0.0000,0.0000,True
1,250.0,7.5000,340.0000,340.0000,-18.3343,50.3732,817.8696,499994.4139,6.5000e+06,-817.8696,0.9000,False
2,500.0,15.0000,340.0000,340.0000,-73.0236,200.6308,1621.7453,499977.7513,6.5001e+06,-1621.7453,0.9000,True
3,750.0,29.7039,325.3217,325.3217,-225.8971,469.1530,2378.7877,499931.1741,6.5001e+06,-2378.7877,1.8745,False
4,1000.0,45.0000,320.0000,320.0000,-529.8050,860.8410,3029.0247,499838.5800,6.5003e+06,-3029.0247,1.8745,True
5,1250.0,52.0800,308.9795,308.9795,-969.0160,1287.7712,3572.6761,499704.7621,6.5004e+06,-3572.6761,1.3033,False
6,1500.0,60.0000,300.0000,300.0000,-1529.7691,1670.0043,4031.1367,499533.9129,6.5005e+06,-4031.1367,1.3033,True
7,1750.0,67.1892,289.4491,289.4491,-2196.1897,1974.5463,4396.4890,499330.8691,6.5006e+06,-4396.4890,1.4236,False
8,2000.0,75.0000,280.0000,280.0000,-2945.4405,2169.9046,4662.5769,499102.5888,6.5007e+06,-4662.5769,1.4236,True


The MD interval example asks for a point every metre, and `interpolate` resamples the trajectory every 100 m of MD while keeping the survey stations. The survey stations are flagged in both.

In [15]:
dense = from_payload(REQUESTS["08_md_interval_interpolation"])
print(f"{len(dense.stations)} stations, {len(dense.stations_i)} points every metre")
table = dense.stations_i.to_dataframe()
table[table["md"].isin([0, 1, 50, 99, 100, 101, 150, 200])]

3 stations, 201 points every metre


,md,inclination,azimuth_true,azimuth_grid,east,north,tvd,x,y,z,dls,is_survey_station
0,0.0,0.0000,338.5294,340.0000,0.0000,0.0000,0.0000,400000.0000,6.5000e+06,0.0000,0.0000,True
1,1.0,0.1000,318.5294,320.0000,-0.0006,0.0007,1.0000,399999.9994,6.5000e+06,-1.0000,3.0000,False
50,50.0,5.0000,318.5294,320.0000,-1.4439,1.6337,49.9366,399998.5989,6.5000e+06,-49.9366,3.0000,False
99,99.0,9.9000,318.5294,320.0000,-5.6500,6.3928,98.5081,399994.5174,6.5000e+06,-98.5081,3.0000,False
100,100.0,10.0000,318.5294,320.0000,-5.7644,6.5223,99.4931,399994.4064,6.5000e+06,-99.4931,3.0000,True
101,101.0,10.2162,317.1185,318.5890,-5.8823,6.6523,100.4776,399994.2920,6.5000e+06,-100.4776,9.8608,False
150,150.0,24.1165,286.7166,288.1872,-18.5100,12.7561,147.2591,399981.8285,6.5000e+06,-147.2591,9.8608,False
200,200.0,40.0000,278.5294,280.0000,-44.3626,18.1145,189.5183,399956.1291,6.5000e+06,-189.5183,9.8608,True


## 8. A geographic CRS

In a geographic CRS the positions are longitude and latitude in degrees, on the CRS's own ellipsoid. There is no grid, so the convergence is zero, and azimuths given against grid north are refused.

In [16]:
body = REQUESTS["05_geographic_lmp_method"]
geographic = from_payload(body).stations
print(*geographic.operations, sep="\n")
display(
    geographic.to_dataframe()[
        ["md", "inclination", "azimuth_true", "east", "north", "tvd", "x", "y", "z"]
    ]
)

try:
    from_payload(body | {"azimuthReference": "GN"})
except UnsupportedCRSError as error:
    print("refused:", error)

trajectory CRS EPSG:4326; wellhead at (4, 58), elevation 0 m
azimuths against true north
minimum curvature over 5 survey stations
integrated offsets over WGS 84's ellipsoid at each step's elevation
converted from WGS 84 to WGS 84


,md,inclination,azimuth_true,east,north,tvd,x,y,z
0,0.0,0.0,45.0,0.0000,0.0000,0.0000,4.0000,58.0000,0.0000
1,1000.0,30.0,45.0,180.9293,180.9293,954.9297,4.0031,58.0016,-954.9297
2,2000.0,60.0,45.0,675.2372,675.2372,1653.9867,4.0114,58.0061,-1653.9867
3,3000.0,75.0,45.0,1326.6547,1326.6547,2035.5782,4.0224,58.0119,-2035.5782
4,5000.0,75.0,45.0,2692.6801,2692.6801,2553.2163,4.0456,58.0242,-2553.2163


refused: azimuths are given against grid north, but WGS 84 is geographic and has no grid; give them against true north


## 9. Moving to WGS 84

Every position is computed on the trajectory CRS's own datum. Moving it to another datum goes through `geodetic_engine.geodesy.Transformation`, so the datum shift has to be named, or be part of the CRS as a bound CRS, exactly as for any other transformation.

The deviated well's survey is placed here in ED50 / UTM zone 31N.

In [17]:
from pyproj import CRS
from pyproj.crs import BoundCRS, CoordinateOperation

body = REQUESTS["02_deviated_well_build"]
ed50 = compute_trajectory(survey_of(body), wellhead_of(body), "EPSG:23031", north="GN")

try:
    ed50.to_geographic()
except AmbiguousOperationError as error:
    print("refused:", str(error)[:110], "...")

named = ed50.to_geographic(operation="EPSG:1133")
print("\nwith EPSG:1133 named:", named.operation.name)

bound = BoundCRS(
    CRS("EPSG:23031"), CRS("EPSG:4326"), CoordinateOperation.from_epsg(1133)
)
by_bound = compute_trajectory(
    survey_of(body), wellhead_of(body), bound, north="GN"
).to_geographic()
print("through a bound CRS, no operation named:", by_bound.operation.name)
print(
    "same longitudes and latitudes:",
    np.allclose(named.coordinates.to_numpy(), by_bound.coordinates.to_numpy()),
)
named.coordinates.to_dataframe()

refused: EPSG:23031 to EPSG:4326 involves a datum change; every datum operation must be named explicitly. allow_any_ope ...

with EPSG:1133 named: ED50 to WGS 84 (1)
through a bound CRS, no operation named: ED50 to WGS 84 (1)
same longitudes and latitudes: True


,Lon,Lat
0,2.9984,59.5364
1,2.9984,59.5364
2,2.9988,59.5367
3,3.0003,59.5374
4,3.0025,59.5385
5,3.0078,59.5412
6,3.0167,59.5457


## 10. Plotting

`plot_trajectory()`, or `WellTrajectory.plot()`, builds a plotly figure. Each example is shown two ways:

- **In the notebook**: a figure left as the last line of a cell is drawn below it, at a fixed size.
- **In the browser**: `open_in_browser()` opens the same figure in a new tab that fills the window. In the dev container that is the host's browser, through VS Code's port forwarding. The address is printed too, to open by hand.

Paths are drawn along their arcs, coloured by dogleg severity or by MD, with faint shadows on the floor and two walls to make depth readable. All three axes share one scale, in metres on the ground, so a vertical well is drawn vertical and a build at 45° looks like 45°. An axis the wells barely span is widened around them rather than stretched.

| Action | Control |
| --- | --- |
| Rotate | drag with the left button |
| Pan | drag with the right button, or the toolbar's pan tool |
| Zoom | scroll |
| Inspect a point | hover: MD, angles, TVD, DLS and position |
| Hide a well | click its name in the legend |
| Back to the start | double-click, or the toolbar's home button |

In [18]:
s_curve = from_payload(REQUESTS["04_s_curve_with_turn"]).stations
figure = s_curve.plot(color_by="dls", labels=["S-curve with turn"])
figure

In [19]:
print(open_in_browser(figure))

http://127.0.0.1:39233/WGS-84-UTM-zone-31N-1.html


Several wells in the same CRS share one set of axes. The vertical, deviated and extended reach wells all start at the same wellhead.

In [20]:
wells = {
    "Vertical": "01_simple_vertical_well",
    "Deviated": "02_deviated_well_build",
    "Extended reach": "07_extended_reach_well",
}
one_wellhead = [from_payload(REQUESTS[name]).stations for name in wells.values()]
figure = plot_trajectory(
    *one_wellhead,
    color_by=None,
    labels=list(wells),
    title="Three wells from one wellhead",
)
figure

In [21]:
print(open_in_browser(figure))

http://127.0.0.1:39233/Three-wells-from-one-wellhead-2.html


In [22]:
figure = by_method[Method.LMP].plot(
    color_by="md", title="Extended reach well, coloured by MD"
)
figure

In [23]:
print(open_in_browser(figure))

http://127.0.0.1:39233/Extended-reach-well-coloured-by-MD-3.html


The figure is a plain plotly figure: restyle it with `figure.update_layout(...)`, write it to a self-contained HTML file to keep or share with `figure.write_html(path)`, or to a still PNG, SVG or PDF with `figure.write_image(path)`.